### Dataset Preparation

### Preprocessing

In [2]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()                              # ubah jadi huruf kecil
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)  # hapus URL
    text = re.sub(r"@\w+", "", text)                 # hapus mention
    text = re.sub(r"#\w+", "", text)                 # hapus hashtag
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)      # hapus simbol & tanda baca
    text = re.sub(r"\s+", " ", text).strip()         # hapus spasi berlebih
    return text


### Load dan Tokenisasi IndoBERT

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)


d:\Langs\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Fine-Tuning IndoBERT

In [5]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Muat dataset
df = pd.read_csv("datasets/yt_6500.csv")
df.head(25)

,text,label
0,CANTIKA ABIGAIL TAU MEDAN MAGNET PASUKAN 😂,0
1,Tau curi umur juga 😆,0
2,Hahhahaahaa,0
3,ibunya cantika dulu artis penyanyi...lagunya h...,0
4,Ini sih gilaaa yaaa😂😂😂😂,0
5,KELUARKAN EPISODE KETUA NAGA HITAM KAMI,0
6,Sesepu joglonya @dustin tiffani,0
7,"Dia ketawa aja merdu, pake nada pula 😅 Minder ...",0
8,Gue denger2in jegel ni kalo mau nanya kebanyak...,0
9,"wak.. mak ber,&harahap lucu juga kali kalau di...",0


In [6]:
df["text"] = df["text"].apply(clean_text)
df.head(25)

,text,label
0,cantika abigail tau medan magnet pasukan,0
1,tau curi umur juga,0
2,hahhahaahaa,0
3,ibunya cantika dulu artis penyanyi lagunya hit...,0
4,ini sih gilaaa yaaa,0
5,keluarkan episode ketua naga hitam kami,0
6,sesepu joglonya tiffani,0
7,dia ketawa aja merdu pake nada pula minder bgt...,0
8,gue denger2in jegel ni kalo mau nanya kebanyak...,0
9,wak mak ber harahap lucu juga kali kalau di un...,0


In [11]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)
tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)
train_dataset = tokenized_dataset["train"]
eval_dataset = tokenized_dataset["test"]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Map: 100%|██████████| 6285/6285 [00:00<00:00, 42180.76 examples/s]


In [12]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 5. Model IndoBERT
model = AutoModelForSequenceClassification.from_pretrained(
    "indobenchmark/indobert-base-p1",
    num_labels=2
)

# 6. Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

# 7. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

# 8. Mulai training
trainer.train()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.















































































                                        

                                         
  0%|          | 0/1887 [20:09<?, ?it/s]         



{'eval_loss': 0.09567457437515259, 'eval_runtime': 39.0267, 'eval_samples_per_second': 32.209, 'eval_steps_per_second': 2.024, 'epoch': 1.0}


                                        
  0%|          | 0/1887 [27:19<?, ?it/s]         

{'loss': 0.1111, 'grad_norm': 0.014647467993199825, 'learning_rate': 9.417989417989418e-06, 'epoch': 1.59}

















































































                                        
                                              

  0%|          | 0/1887 [33:01<?, ?it/s]       



{'eval_loss': 0.0805801972746849, 'eval_runtime': 39.208, 'eval_samples_per_second': 32.06, 'eval_steps_per_second': 2.015, 'epoch': 2.0}

















































































                                        
                                              

  0%|          | 0/1887 [45:54<?, ?it/s]       

                                        
100%|██████████| 945/945 [39:12<00:00,  2.49s/it]

{'eval_loss': 0.09065866470336914, 'eval_runtime': 38.3657, 'eval_samples_per_second': 32.764, 'eval_steps_per_second': 2.059, 'epoch': 3.0}
{'train_runtime': 2352.4964, 'train_samples_per_second': 6.412, 'train_steps_per_second': 0.402, 'train_loss': 0.07253447033110119, 'epoch': 3.0}


TrainOutput(global_step=945, training_loss=0.07253447033110119, metrics={'train_runtime': 2352.4964, 'train_samples_per_second': 6.412, 'train_steps_per_second': 0.402, 'total_flos': 309741323257200.0, 'train_loss': 0.07253447033110119, 'epoch': 3.0})

### Evaluasi Model

In [13]:
predictions = trainer.predict(eval_dataset)
preds = predictions.predictions.argmax(-1)

from sklearn.metrics import classification_report
print(classification_report(eval_dataset["label"], preds, target_names=["normal", "judol"]))

100%|██████████| 79/79 [00:38<00:00,  2.06it/s]

              precision    recall  f1-score   support

      normal       0.98      0.98      0.98       602
       judol       0.98      0.98      0.98       655

    accuracy                           0.98      1257
   macro avg       0.98      0.98      0.98      1257
weighted avg       0.98      0.98      0.98      1257



### Testing

In [1]:
text = "slot gacor gampang menang hari ini!"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
outputs = model(**inputs)
prediction = outputs.logits.argmax(-1).item()
print("Judol" if prediction == 1 else "Normal")


NameError: name 'tokenizer' is not defined